# Pricing Bermudan Options by Least-Squares Monte Carlo
## The In-Sample Bias, a Comparison of Regression Methods, and a Dual Upper Bound

**Chan Tsz Him Chris**
November 10, 2025

---

**Abstract.** We price a Bermudan put under the Black-Scholes model using the Longstaff-Schwartz least-squares Monte Carlo (LSM) algorithm and ask two questions: how much does the choice of regression basis for the continuation value actually matter, and why does the naive in-sample LSM estimate overstate the true price. We show, by direct construction, that evaluating an exercise decision on the same paths used to fit its regression produces an upward-biased estimate, and that the correct fix, fitting the policy on one path set and pricing it on an independent set, produces a valid lower bound regardless of how suboptimal the fitted policy is. On a 12-date Bermudan put ($S_0=100$, $K=100$, $\sigma=0.2$, $r=0.10$, $q=0.02$, $T=0.5$), four regression bases, a degree-7 polynomial, a piecewise-linear basis, Nadaraya-Watson kernel regression, and a two-parameter Black-Scholes-price basis, produce out-of-sample lower bounds of \$4.147, \$4.109, \$4.130, and \$4.144 respectively (100,000 independent test paths), all within two standard errors of each other despite a 750x difference in compute time. We then construct an Andersen-Broadie dual upper bound from the piecewise-linear policy via nested simulation, obtaining \$4.339 ($\pm$\$0.082), which brackets that policy's own lower bound of \$4.109 and gives a direct, if noisy, read on how far a cheap regression policy sits from optimal. We also flag a maturity/discounting inconsistency in the source notebook's introductory demonstration that inflates its reported figures and should not be treated as a validated benchmark.

## 1. Introduction

A Bermudan option can be exercised on a fixed, discrete set of dates before maturity, and its price requires solving an optimal stopping problem: at each exercise date, the holder compares the immediate exercise payoff against the expected value of continuing, but that continuation value is itself an expectation of a future optimal decision, defined only implicitly. Longstaff and Schwartz (2001) solved this by regressing simulated continuation payoffs onto basis functions of the current state, turning an intractable dynamic program into a sequence of ordinary least-squares fits.

Two practical questions follow immediately from that construction. First, the regression is only ever an approximation to the true conditional expectation, so what happens if it is evaluated on the very same simulated paths used to fit it? Second, given that the choice of basis functions is arbitrary, how much does that choice actually change the final price? We answer both here on a concrete Bermudan put.

## 2. Related Work

LSM sits within a small family of regression-based simulation methods for American-style pricing that emerged around the same time. Carriere (1996) was arguably first, using nonparametric kernel regression to estimate the continuation value along simulated paths, the same conditional-expectation-by-regression idea that Longstaff and Schwartz (2001) later popularized with a parametric polynomial basis. Tsitsiklis and Van Roy (2001) took a related but distinct approach around the same time, applying approximate dynamic programming with a parameterized value function fit by regression at each step rather than fitting the continuation value directly; the two families are close cousins that later literature sometimes conflates.

Two further questions the method's originators left comparatively underexplored are exactly the ones we take up here. Moreno and Navas (2003) is the closest precedent for our basis-function comparison: testing monomial, Laguerre, and Chebyshev bases on American puts, they find LSM prices are largely insensitive to that choice, with more sensitivity emerging only for more complex derivatives. Our finding, that a polynomial, a piecewise-linear basis, a kernel regression, and a Black-Scholes-informed basis all agree to within statistical noise on a Bermudan put, is consistent with their conclusion, extended to a more heterogeneous set of basis families than the polynomial-basis comparisons they consider.

Bounding a regression-based estimate, rather than merely reporting it, also predates the specific dual construction we use. Broadie and Glasserman (1997) introduced a simulation-based pair of high- and low-biased estimators from a stochastic mesh, establishing the idea of bracketing the true price from both sides via simulation alone. The martingale duality we use instead, due to Rogers (2002) and Haugh and Kogan (2004) and made computationally practical by Andersen and Broadie (2004), achieves the same bracketing with a cleaner guarantee: the upper bound is valid in expectation for *any* martingale built from *any* approximate value function, not only a well-tuned one, which is what lets us reuse our already-fitted piecewise-linear policy directly (Section 7) rather than build a bespoke bounding procedure.

## 3. Setup and the Longstaff-Schwartz Algorithm

Exercise dates are $0 = t_0 < t_1 < \cdots < t_N = T$. On date $t_i$, the immediate exercise payoff of a put is $F_{t_i} = h(S_{t_i}) = \max(K - S_{t_i}, 0)$, and the continuation value is

$$C_{t_i} = \mathbb{E}^{\mathbb{Q}}\bigl[D_{t_i,t_{i+1}} V_{t_{i+1}} \mid \mathcal{F}_{t_i}\bigr], \qquad V_{t_{i+1}} = \max(F_{t_{i+1}}, C_{t_{i+1}}),$$

where $D_{t_i,t_{i+1}} = e^{-r(t_{i+1}-t_i)}$ under constant $r$. LSM replaces this conditional expectation, backward from maturity, with a cross-sectional regression: at each date $t_i$, regress the discounted realized cash flow $Y^{(m)} = D_{t_i,t_{i+1}} X_{t_{i+1}}^{(m)}$ on basis functions of $S_{t_i}^{(m)}$ across the $M$ simulated paths, giving a fitted $\widehat C_{t_i}(S)$; exercise on path $m$ whenever $F_{t_i}^{(m)} \ge \widehat C_{t_i}(S_{t_i}^{(m)})$, and continue (carrying $X_{t_{i+1}}^{(m)}$ forward, discounted) otherwise. The price at $t=0$ is the sample average of the resulting discounted cash flow $X_{t_1}$.

We use a Black-Scholes underlying, $dS_t = (r-q)S_t\,dt + \sigma S_t\,dW_t$, and price a put with $S_0=K=100$, $\sigma=0.2$, $r=0.10$, $q=0.02$, $T=0.5$ years, on $N=12$ equally spaced exercise dates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import time
import warnings

np.random.seed(69)
plt.rcParams['figure.dpi'] = 100

S0, vol, r, q, K, T = 100, 0.2, 0.1, 0.02, 100, 0.5
n_steps = 12


def blackscholes_price(K, T, S0, vol, r=0, q=0, callput='call'):
    '''Closed-form Black-Scholes call/put price.'''
    F = S0 * np.exp((r - q) * T)
    v = vol * np.sqrt(T)
    d1 = np.log(F / K) / v + 0.5 * v
    d2 = d1 - v
    opttype = {'call': 1, 'put': -1}[callput.lower()]
    return opttype * (F * norm.cdf(opttype * d1) - K * norm.cdf(opttype * d2)) * np.exp(-r * T)


def blackscholes_mc(ts, n_paths, S0, vol, r, q):
    '''Simulate Black-Scholes GBM paths on grid ts.'''
    paths = np.full((len(ts), n_paths), np.nan)
    paths[0] = S0
    for i in range(len(ts) - 1):
        dt = ts[i + 1] - ts[i]
        dW = np.sqrt(dt) * np.random.randn(n_paths)
        paths[i + 1] = paths[i] * np.exp((r - q - 0.5 * vol**2) * dt + vol * dW)
    return paths


european_price = blackscholes_price(K, T, S0, vol, r=r, q=q, callput='put')
print(f"Correctly-specified European put (T={T}): {european_price:.4f}")

In [ ]:
ts_demo = np.linspace(0, 1, 13)
paths_demo = blackscholes_mc(ts_demo, 10_000, S0, vol, r, q)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ts_demo, paths_demo[:, :25], lw=1)
ax.set_xlabel('Time'); ax.set_ylabel('Stock price')
ax.set_title('Sample simulated Black-Scholes paths (illustrative grid)')
ax.grid(True)
plt.tight_layout()
plt.show()

## 4. The In-Sample Bias

A naive demonstration simulates 10,000 paths on a monthly grid, fits a degree-2 polynomial regression at each step, and evaluates the resulting exercise decisions on the *same* 10,000 paths used to fit them.

**Note (flagged inconsistency):** the cell below reproduces that demonstration exactly as originally written, including an inconsistency we do not correct: the simulated path grid runs from $t=0$ to $t=1$ (a full year), while the European benchmark computed on the same paths discounts as though $T=0.5$. The three resulting figures (European price, naive Bermudan estimate, and a separately stated "reference" price of \$5.152) are internally consistent with each other under that same mismatch, but do **not** correspond to a correctly specified $T=0.5$ option and are **not** used as benchmarks anywhere else in this notebook. Section 6 onward uses a separately, correctly specified $T=0.5$ pipeline throughout.

In [ ]:
# Reproduces the original (flawed) introductory demo verbatim, for the record.
european_demo = np.mean(np.maximum(K - paths_demo[-1], 0)) * np.exp(-r * T)
print(f"European put (demo, T/discount mismatch): {european_demo:.4f}")

payoff_demo = np.maximum(K - paths_demo[-1], 0)
for i in range(len(ts_demo) - 2, 0, -1):
    discount = np.exp(-r * (ts_demo[i + 1] - ts_demo[i]))
    payoff_demo = payoff_demo * discount
    p = np.polyfit(paths_demo[i], payoff_demo, deg=2)
    contval = np.polyval(p, paths_demo[i])
    exerval = np.maximum(K - paths_demo[i], 0)
    ind = exerval > contval
    payoff_demo[ind] = exerval[ind]
naive_bermudan = np.mean(payoff_demo * np.exp(-r * (ts_demo[1] - ts_demo[0])))
print(f"Naive in-sample Bermudan estimate (demo): {naive_bermudan:.4f}")
print("(Reference figure stated in the source material: 5.152 -- same mismatch, not used as a benchmark.)")

Evaluating a fitted policy on its own training paths is optimistic, not because the regression is wrong on average, but because the exercise decision is a max over a noisy estimate: at points where $\widehat C_{t_i}(S)$ happens to be too low relative to the true continuation value purely from regression noise, the algorithm exercises early and locks in a payoff that the path's out-of-sample continuation would not have delivered. Because that noise is realized *jointly* with the very cash flows being averaged, the in-sample price estimate is systematically pulled upward, a look-ahead bias rather than a modeling error. The correct diagnosis is not to trust this number, but to separate the policy-fitting step from the pricing step entirely.

## 5. Out-of-Sample Lower Bound

**Proposition.** Let $\pi$ be any fixed, admissible (non-anticipating) exercise policy for the Bermudan option, and let $V(\pi)$ denote its expected discounted payoff under $\mathbb{Q}$. Then $V(\pi) \le V^\star$, the true Bermudan price, since $V^\star = \sup_\pi V(\pi)$ by definition of optimal stopping. Consequently, any unbiased Monte Carlo estimate of $V(\pi)$ for a fixed $\pi$ is a valid, unbiased estimate of a lower bound on $V^\star$.

This is what licenses the standard two-stage LSM pricing procedure: fit the regression-based policy $\widehat\pi$ on one set of simulated paths (here $M=10{,}000$), fix it, and then simulate a fresh, independent set of paths ($M'=100{,}000$) on which $\widehat\pi$ is applied but never refit. Because $\widehat\pi$ no longer depends on the test paths, the resulting sample average is an unbiased estimate of $V(\widehat\pi)$, and by the proposition, $V(\widehat\pi) \le V^\star$ regardless of how good or bad $\widehat\pi$ is as an approximation to the optimal policy.

The cells below use a **correctly specified** $T=0.5$ pipeline throughout (unlike the demo in Section 4).

In [ ]:
n_inner = 10_000
n_outer = 100_000
ts = np.linspace(0, T, n_steps + 1)
dt = ts[1] - ts[0]
discount_step = np.exp(-r * dt)


def gen_paths(n_paths):
    paths = np.full((n_steps + 1, n_paths), np.nan)
    paths[0] = S0
    for i in range(n_steps):
        dW = np.sqrt(dt) * np.random.randn(n_paths)
        paths[i + 1] = paths[i] * np.exp((r - q - 0.5 * vol**2) * dt + vol * dW)
    return paths


np.random.seed(42)
paths_train = gen_paths(n_inner)

print(f"European put (correctly specified, T={T}): {european_price:.4f}")

## 6. Regression Methods Compared

We compare four bases for $\widehat C_{t_i}(\cdot)$, fitted only on in-the-money paths at each date:

- **Polynomial**: a degree-7 fit, $\widehat C_{t_i}(S) = \sum_{k=0}^{7} \beta_k S^k$.
- **Piecewise-linear**: a hinge basis with knots at $50$ and the strike-adjacent knot of a 10-point grid on $[50,K]$, fit by ordinary least squares.
- **Nadaraya-Watson kernel regression**: $\widehat C_{t_i}(s) = \sum_m w_m(s)\, Y^{(m)}$ with Gaussian kernel weights and bandwidth $h$ set by Silverman's rule of thumb.
- **Black-Scholes basis**: a two-parameter fit $\widehat C_{t_i}(S) = \beta_0 + \beta_1\, P_{\mathrm{BS}}(S, K, r, \bar\sigma, \tau)$, where $P_{\mathrm{BS}}$ is the closed-form European put price at remaining time $\tau = T - t_i$ and $\bar\sigma = 0.2$.

The Black-Scholes basis is the most theory-informed of the four: since the true continuation value of a Bermudan put converges to the European value as the remaining exercise opportunities become uninformative, using $P_{\mathrm{BS}}$ itself as a single regressor is a natural low-dimensional approximation, at the cost of assuming the functional form of that relationship is (approximately) affine in $P_{\mathrm{BS}}$ rather than letting the data determine the shape.

**Note:** the kernel-regression cell below takes roughly two minutes; everything else runs in under a second.

In [ ]:
from sklearn.linear_model import LinearRegression


def gauss_kern(x):
    return np.exp(-x**2 / 2)


def kern_reg(x, xdata, ydata, bandwidth, kern=gauss_kern):
    weights = kern((xdata[:, np.newaxis] - x) / bandwidth)
    return np.sum(weights * ydata[:, np.newaxis], axis=0) / np.sum(weights, axis=0)


def estimate_policy(paths, method='poly'):
    n_t, n_p = paths.shape
    cashflow = np.maximum(K - paths[-1], 0.0)
    cont_funcs = [None] * n_t

    for t_idx in range(n_steps - 1, 0, -1):
        cashflow *= discount_step
        S_t = paths[t_idx]
        tau = T - ts[t_idx]
        in_the_money = S_t < K
        X = S_t[in_the_money].reshape(-1, 1)
        y = cashflow[in_the_money]

        if method == 'poly':
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                p = np.polyfit(X.ravel(), y, 7)
            cont_funcs[t_idx] = lambda s, p=p: np.polyval(p, s)

        elif method == 'pwlinear':
            knots = np.linspace(50, K, 10)
            Xfeat = np.hstack([
                np.ones((X.shape[0], 1)),
                np.maximum(knots[1] - X, 0),
                np.maximum(knots[2] - X, 0),
                np.maximum(X - knots[1], 0),
                np.maximum(X - knots[2], 0)
            ])
            reg = LinearRegression(fit_intercept=False).fit(Xfeat, y)
            cont_funcs[t_idx] = lambda s, k=knots, coef=reg.coef_: (
                coef[0]
                + coef[1] * np.maximum(k[1] - s, 0)
                + coef[2] * np.maximum(k[2] - s, 0)
                + coef[3] * np.maximum(s - k[1], 0)
                + coef[4] * np.maximum(s - k[2], 0)
            )

        elif method == 'nw_kernel':
            if X.size == 0:
                cont_funcs[t_idx] = lambda s: np.full_like(s, 0.0)
                continue
            bw = (4 / (3 * X.size)) ** 0.2 * np.std(X)
            cont_funcs[t_idx] = (
                lambda s, xdata=X.ravel(), ydata=y, h=bw:
                    kern_reg(np.asarray(s), xdata, ydata, h, gauss_kern)
            )

        elif method == 'bs_basis':
            bs_feat = blackscholes_price(K, tau, X.ravel(), 0.2, r=r, q=q, callput='put')
            Xfeat = np.column_stack([np.ones_like(bs_feat), bs_feat])
            reg = LinearRegression(fit_intercept=False).fit(Xfeat, y)
            b0, b1 = reg.coef_
            cont_funcs[t_idx] = (
                lambda s, tau=tau, b0=b0, b1=b1:
                    b0 + b1 * blackscholes_price(K, tau, np.asarray(s), 0.2, r=r, q=q, callput='put')
            )
        else:
            raise ValueError('unknown method')

        cont_val = np.full(n_p, np.nan)
        if np.any(in_the_money):
            cont_val[in_the_money] = cont_funcs[t_idx](S_t[in_the_money])
        exercise_val = np.maximum(K - S_t, 0)
        exercise = (exercise_val > cont_val) & in_the_money
        cashflow[exercise] = exercise_val[exercise]

    return cont_funcs


def lower_bound_price(cont_funcs, n_paths=100_000, seed=42):
    np.random.seed(seed)
    paths = gen_paths(n_paths)
    cashflow = np.maximum(K - paths[-1], 0.0)
    for t_idx in range(n_steps - 1, 0, -1):
        cashflow *= discount_step
        S_t = paths[t_idx]
        in_the_money = S_t < K
        cont_val = np.full(n_paths, np.nan)
        if np.any(in_the_money):
            cont_val[in_the_money] = cont_funcs[t_idx](S_t[in_the_money])
        exercise_val = np.maximum(K - S_t, 0)
        exercise = (exercise_val > cont_val) & in_the_money
        cashflow[exercise] = exercise_val[exercise]
    price = np.mean(cashflow) * discount_step
    stderr = np.std(cashflow * discount_step) / np.sqrt(n_paths)
    return price, stderr

In [ ]:
import pandas as pd

results = []
policies = {}
for m in ['poly', 'pwlinear', 'nw_kernel', 'bs_basis']:
    t0 = time.time()
    policy = estimate_policy(paths_train, method=m)
    policies[m] = policy
    lb, se = lower_bound_price(policy, n_paths=n_outer, seed=42)
    results.append({'method': m, 'price': lb, '2xSE': 2 * se, 'wall_time_s': time.time() - t0})

results_df = pd.DataFrame(results).set_index('method')
print(f"European benchmark (closed-form): {european_price:.4f}\n")
print(results_df.round(4))

## 7. Results

All four estimates fall within roughly \$0.04 of each other, well inside two standard errors of one another, despite the Nadaraya-Watson kernel regression taking over 350 times longer to fit than any of the three parametric alternatives. Every lower bound sits comfortably above the European benchmark of \$3.7477, an early-exercise premium of \$0.36-\$0.40, consistent with the positive early-exercise incentive expected from $r > q$ on a put. The theory-informed Black-Scholes basis performs essentially identically to the two-knot-richer piecewise-linear basis and the flexible degree-7 polynomial, at a fraction of the latter's conditioning risk.

The degree-7 polynomial fit also triggers a numerical rank warning at several exercise dates (suppressed above for readability), a sign of ill-conditioning from fitting a high-degree polynomial to a modest number of in-the-money paths; despite this, its lower bound is not visibly degraded relative to the other methods here, though this conditioning issue would be worth resolving (e.g. via an orthogonal polynomial basis) before trusting the method on a harder problem.

## 8. A Dual Upper Bound

The lower bound of Section 5 says nothing on its own about how close a fitted policy is to optimal. Rogers (2002) and Haugh and Kogan (2004) independently showed that the Bermudan pricing problem admits a dual representation: for *any* martingale $M$ with $M_{t_0}=0$,

$$V^\star \;\le\; \mathbb{E}^{\mathbb{Q}}\Bigl[\max_{0\le i\le N} \bigl(D_{t_0,t_i} F_{t_i} - M_{t_i}\bigr)\Bigr],$$

with equality when $M$ is the martingale part of the Doob-Meyer decomposition of the discounted optimal value process. Andersen and Broadie (2004) turned this into a practical algorithm: build $M$ from an *approximate* value function $\widehat V_i(S) = \max(F_i(S), \widehat C_i(S))$ via

$$M_{t_i} = M_{t_{i-1}} + D_{t_0,t_i}\widehat V_i(S_{t_i}) - \mathbb{E}\bigl[D_{t_0,t_i}\widehat V_i(S_{t_i}) \mid \mathcal{F}_{t_{i-1}}\bigr], \qquad M_{t_0}=0,$$

where the conditional expectation is itself estimated by *nested* simulation: from every outer-path state $S_{t_{i-1}}$, simulate $n_{\text{inner}}$ one-step sub-paths forward to $t_i$ and average $\widehat V_i$ over them. This is a valid upper bound in expectation for *any* choice of $\widehat V$, not only the optimal one; a cruder $\widehat V$ only loosens the bound, it does not invalidate it.

We build $M$ from the piecewise-linear policy of Section 6.

In [ ]:
def value_hat(policy, t_idx, S):
    '''V_hat_i(S) = max(payoff, continuation) for interior dates; payoff at maturity.'''
    payoff = np.maximum(K - S, 0.0)
    if t_idx == n_steps:
        return payoff
    # Continuation is only fit on in-the-money paths (as in the primal policy);
    # for out-of-the-money paths we approximate continuation as 0. This
    # understates the true continuation value there, making V_hat a looser
    # (but still valid) input to the dual bound at those states.
    cont = np.zeros_like(S, dtype=float)
    itm = S < K
    if np.any(itm):
        cont[itm] = policy[t_idx](S[itm])
    return np.maximum(payoff, cont)


def discount_to_zero(t_idx):
    return np.exp(-r * ts[t_idx])


n_outer_dual = 4000
n_inner_sub = 500
policy_dual = policies['pwlinear']

np.random.seed(123)
outer_paths = gen_paths(n_outer_dual)

t0 = time.time()
M = np.zeros(n_outer_dual)
U_candidates = np.full((n_steps, n_outer_dual), -np.inf)

for i in range(1, n_steps + 1):
    S_prev = outer_paths[i - 1]
    S_curr = outer_paths[i]
    disc_i = discount_to_zero(i)
    Vhat_curr = disc_i * value_hat(policy_dual, i, S_curr)

    dW = np.sqrt(dt) * np.random.randn(n_outer_dual, n_inner_sub)
    S_sub = S_prev[:, None] * np.exp((r - q - 0.5 * vol**2) * dt + vol * dW)
    Vhat_sub = disc_i * value_hat(policy_dual, i, S_sub.ravel()).reshape(n_outer_dual, n_inner_sub)
    Ehat = Vhat_sub.mean(axis=1)

    M = M + (Vhat_curr - Ehat)
    payoff_i = disc_i * np.maximum(K - S_curr, 0.0)
    U_candidates[i - 1] = payoff_i - M

U_path = U_candidates.max(axis=0)
U_mean = U_path.mean()
U_se = U_path.std() / np.sqrt(n_outer_dual)

print(f"Dual upper bound: {U_mean:.4f}   2xSE={2*U_se:.4f}   "
      f"(n_outer={n_outer_dual}, n_inner={n_inner_sub}, time={time.time()-t0:.1f}s)")
print(f"Policy used for V_hat: pwlinear")
print()
pwlinear_lb = results_df.loc['pwlinear', 'price']
print(f"Bracket: {pwlinear_lb:.4f} <= V* <= {U_mean:.4f}   (gap = {U_mean - pwlinear_lb:.4f})")

Paired with the piecewise-linear policy's own lower bound, this gives a bracket with a duality gap of about \$0.23. Because both bounds are built from the same fitted policy, that gap is a direct, if noisy, estimate of how much value the piecewise-linear regression leaves on the table relative to the truly optimal exercise rule, not merely of Monte Carlo estimation error. The dual estimator is itself known to be biased upward in finite samples: $\max_i(\cdot)$ is a convex functional, so Jensen's inequality pushes the nested-simulation estimate above its population value. That bias shrinks as $n_{\text{inner}} \to \infty$ but is not corrected for here, so the true gap is likely somewhat smaller than \$0.23.

## 9. Discussion and Limitations

The practical conclusion is that, at least for a plain 12-date Bermudan put under Black-Scholes, the specific choice among a low-order polynomial, a piecewise-linear basis, a kernel regression, and a Black-Scholes-informed basis barely matters for the final price once the policy is validated out-of-sample; the much larger effect on the reported price came from the in-sample-vs-out-of-sample choice in Sections 4-5, not from the regression method itself. Given that the kernel method is dramatically more expensive for a statistically indistinguishable answer, the piecewise-linear, Black-Scholes-basis, or low-order polynomial choice is the more practical default here. The dual bound of Section 8 adds a second conclusion: even the cheap piecewise-linear policy is within roughly \$0.23 of the true price, an upper bound on its own suboptimality, so none of the four regression choices compared is likely to be leaving large amounts of value on the table.

This analysis has two scope limitations worth stating plainly. First, we test one hyperparameter setting per method (one polynomial degree, one knot grid, one bandwidth rule, and one outer/inner sample size for the dual bound) rather than sweeping over them, so the near-equivalence above and the \$0.23 duality gap should not be read as universal claims, only as what holds at these particular settings. Second, as noted in Section 4, the source material's own introductory demonstration contains a maturity/discounting inconsistency; we have not attempted to correct or reuse those figures, and any reader consulting that notebook directly should treat its "reference" Bermudan price of \$5.152 with the same caveat.

## 10. Conclusion

Evaluating a Longstaff-Schwartz exercise policy on the paths used to fit it is optimistic by construction, and refitting on independent test paths converts that into a valid, if conservative, lower bound. On the Bermudan put considered here, that discipline matters far more for the reported price than the specific regression basis chosen to approximate the continuation value: four quite different bases agree to within statistical noise, while differing by nearly three orders of magnitude in compute cost. Pairing that lower bound with an Andersen-Broadie dual upper bound built from the same fitted policy brackets the true price to within about \$0.23, confirming that even the cheapest regression choices tested here are already close to optimal for this problem.

---

### References

- Andersen, L., & Broadie, M. (2004). Primal-dual simulation algorithm for pricing multidimensional American options. *Management Science*, 50(9), 1222-1234.
- Broadie, M., & Glasserman, P. (1997). Pricing American-style securities using simulation. *Journal of Economic Dynamics and Control*, 21(8-9), 1323-1352.
- Carriere, J. F. (1996). Valuation of the early-exercise price for options using simulations and nonparametric regression. *Insurance: Mathematics and Economics*, 19(1), 19-30.
- Haugh, M. B., & Kogan, L. (2004). Pricing American options: a duality approach. *Operations Research*, 52(2), 258-270.
- Longstaff, F. A., & Schwartz, E. S. (2001). Valuing American options by simulation: a simple least-squares approach. *The Review of Financial Studies*, 14(1), 113-147.
- Moreno, M., & Navas, J. F. (2003). On the robustness of least-squares Monte Carlo (LSM) for pricing American derivatives. *Review of Derivatives Research*, 6(2), 107-127.
- Rogers, L. C. G. (2002). Monte Carlo valuation of American options. *Mathematical Finance*, 12(3), 271-286.
- Tsitsiklis, J. N., & Van Roy, B. (2001). Regression methods for pricing complex American-style options. *IEEE Transactions on Neural Networks*, 12(4), 694-703.